<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/APAC_AGENTIC_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/frank-morales2020/AST/blob/main/Qwen3dot8_27B_topo_cbp_gguf.ipynb

## setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [1]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [2]:
import os
import struct
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/GGUF_Models")

def read_gguf_header(filepath):
    try:
        with open(filepath, "rb") as f:
            magic = f.read(4)
            if magic != b"GGUF":
                return {"error": "Invalid magic bytes"}
            version = struct.unpack("<I", f.read(4))[0]
            tensor_count = struct.unpack("<Q", f.read(8))[0]
            kv_count = struct.unpack("<Q", f.read(8))[0]
            return {
                "version": version,
                "tensor_count": tensor_count,
                "kv_count": kv_count
            }
    except Exception as e:
        return {"error": str(e)}

print(f"{'Model Filename':<52} | {'Size (GB)':<10} | {'GGUF Version':<12} | {'Status'}")
print("-" * 95)

total_size = 0
for file_path in sorted(model_dir.glob("*.gguf")):
    file_size = file_path.stat().st_size
    size_gb = file_size / (1024**3)
    total_size += file_size

    meta = read_gguf_header(file_path)
    if "error" not in meta:
        status = "Verified"
        version_str = f"v{meta['version']}"
    else:
        status = meta["error"]
        version_str = "N/A"

    print(f"{file_path.name:<52} | {size_gb:>8.2f}GB | {version_str:<12} | {status}")

print("-" * 95)
print(f"Total Portfolio Storage: {total_size / (1024**3):.2f} GB across {len(list(model_dir.glob('*.gguf')))} models.")

Model Filename                                       | Size (GB)  | GGUF Version | Status
-----------------------------------------------------------------------------------------------
certified-scorpion-topo-qwen-bf16.gguf               |    14.19GB | v3           | Verified
deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf |    69.83GB | v3           | Verified
deepseek_moe_16b_topocbp_q4_k_m.gguf                 |    10.11GB | v3           | Verified
gpt-oss-20b-topo-governed-Q4_K_M.gguf                |    14.72GB | v3           | Verified
llama-3.1-70b-topo-governed-q4_k_m.gguf              |    39.60GB | v3           | Verified
mixtral_8x22b_q4_k_m.gguf                            |    79.71GB | v3           | Verified
muse-glimmer-30b-card-faithful-Q4_K_M.gguf           |    15.77GB | v3           | Verified
nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf          |     4.97GB | v3           | Verified
qwen3.8-27b-topocbp-Q8_0.gguf                        |    26.63GB | v3       

## DL/ML - APAC GGUF

In [1]:
import os
from llama_cpp import Llama

class HybridAPACAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models"):
        print("Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...")

        # 1. Router: DeepSeek MoE (Smallest ~10.11 GB) -> Full GPU Offload
        print("Loading Router: DeepSeek MoE (n_gpu_layers=-1)...")
        self.router = Llama(
            model_path=os.path.join(model_dir, "deepseek_moe_16b_topocbp_q4_k_m.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 2. Synthesizer: Qwen 3.8-27B (Medium ~26.63 GB) -> Full GPU Offload
        print("Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...")
        self.synthesizer = Llama(
            model_path=os.path.join(model_dir, "qwen3.8-27b-topocbp-Q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 3. Multilingual Specialist: Sarvam-30B (~31.86 GB) -> Full GPU Offload
        print("Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...")
        self.multilingual = Llama(
            model_path=os.path.join(model_dir, "sarvam-30b-topo-governed-q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 4. Foundational: Certified Scorpion Qwen (~14.19 GB) -> Full GPU Offload
        print("Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...")
        self.foundational = Llama(
            model_path=os.path.join(model_dir, "certified-scorpion-topo-qwen-bf16.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 5. Deep Reasoner: DeepSeek-R1 70B (Biggest ~69.83 GB) -> Partial Offload (n_gpu_layers=35)
        print("Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...")
        self.reasoner = Llama(
            model_path=os.path.join(model_dir, "deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"),
            n_ctx=8192, n_gpu_layers=25, verbose=False
        )

        print("All 5 APAC models active with custom offloading profiles under Topo-CBP governance.")

    def route_task(self, query: str) -> str:
        prompt = (
            f"Classify the following query into one of five categories "
            f"[REASONING, SYNTHESIS, MULTILINGUAL, FOUNDATIONAL, GENERAL]:\n\n{query}\nCategory:"
        )
        res = self.router(prompt, max_tokens=20, temperature=0.1)
        category = res["choices"][0]["text"].strip().upper()
        for cat in ["REASONING", "SYNTHESIS", "MULTILINGUAL", "FOUNDATIONAL", "GENERAL"]:
            if cat in category:
                return cat
        return "SYNTHESIS"

    def execute_pipeline(self, query: str):
        task_type = self.route_task(query)
        print(f"\n[Router: DeepSeek MoE] Task classified as: {task_type}")

        if task_type == "REASONING":
            print("[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...")
            prompt = f"<|im_start|>system\nYou are a rigorous logical reasoning agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n<think>\n"
            res = self.reasoner(prompt, max_tokens=1536, temperature=0.5, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "SYNTHESIS":
            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25] Executing synthesis...")
            prompt = f"<|im_start|>system\nYou are an expert synthesis agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "MULTILINGUAL":
            print("[Agent: Sarvam-30B (n_gpu_layers=25)] Processing regional context...")
            prompt = f"System: You are an expert multilingual processor.\nUser: {query}\nAssistant:"
            res = self.multilingual(prompt, max_tokens=1024, temperature=0.7)
            return res["choices"][0]["text"].strip()

        elif task_type == "FOUNDATIONAL":
            print("[Agent: Certified Scorpion Qwen (n_gpu_layers=-1)] Applying instruction tuning...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.foundational(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        else:
            print("[Agent: Qwen Synthesizer] Handling general objective...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

# --- Execution Test ---
if __name__ == "__main__":
    system = HybridAPACAgentSystem()
    sample_query = "Prove the convergence of continual backpropagation under topological invariant locking."
    result = system.execute_pipeline(sample_query)

    print("\n==============================")
    print("FINAL HYBRID PIPELINE OUTPUT")
    print("==============================\n")
    print(result)

Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...
Loading Router: DeepSeek MoE (n_gpu_layers=-1)...
Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...
Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...
Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...
Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...
All 5 APAC models active with custom offloading profiles under Topo-CBP governance.

[Router: DeepSeek MoE] Task classified as: REASONING
[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...

FINAL HYBRID PIPELINE OUTPUT

Okay, so I need to prove the convergence of continual backpropagation under topological invariant locking. Hmm, I'm not exactly sure what that means, but I'll try to break it down. 

First, I know that backpropagation is a common algorithm used in training neural networks. It's about adjusting the weights to minimize the error. Continual backpropagation might refer to updating the weights continuo

In [2]:
!nvidia-smi

Sun Nov 15 00:21:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             88W /  600W |   92929MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Medical  - APAC GGUF

In [1]:
import os
from llama_cpp import Llama

class PureMedicalAPACAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models"):
        print("Initializing Pure Medical APAC Agent Ecosystem (95GB VRAM Custom Offloading)...")

        # 1. Router: DeepSeek MoE (Smallest ~10.11 GB) -> Full GPU Offload
        print("Loading Clinical Router: DeepSeek MoE (n_gpu_layers=-1)...")
        self.router = Llama(
            model_path=os.path.join(model_dir, "deepseek_moe_16b_topocbp_q4_k_m.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 2. Synthesizer: Qwen 3.8-27B (Medium ~26.63 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Clinical Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...")
        self.synthesizer = Llama(
            model_path=os.path.join(model_dir, "qwen3.8-27b-topocbp-Q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 3. Multilingual Specialist: Sarvam-30B (~31.86 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Regional Biomedical Specialist: Sarvam-30B (n_gpu_layers=25)...")
        self.multilingual = Llama(
            model_path=os.path.join(model_dir, "sarvam-30b-topo-governed-q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 4. Foundational: Certified Scorpion Qwen (~14.19 GB) -> Full GPU Offload
        print("Loading Foundational Clinical Model: Certified Scorpion Qwen (n_gpu_layers=-1)...")
        self.foundational = Llama(
            model_path=os.path.join(model_dir, "certified-scorpion-topo-qwen-bf16.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 5. Deep Reasoner: DeepSeek-R1 70B (Biggest ~69.83 GB) -> Partial Offload (n_gpu_layers=25)
        print("Loading Deep Clinical Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...")
        self.reasoner = Llama(
            model_path=os.path.join(model_dir, "deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"),
            n_ctx=8192, n_gpu_layers=25, verbose=False
        )

        print("All 5 Medical APAC agent nodes active with custom offloading profiles under Topo-CBP governance.")

    def route_clinical_task(self, case_notes: str) -> str:
        prompt = (
            f"Classify the following clinical query into one of five categories "
            f"[DIFFERENTIAL_DIAGNOSIS, LITERATURE_ANALYSIS, PROTOCOL_SYNTHESIS, FOUNDATIONAL_REVIEW, GENERAL_CLINICAL]:\n\n{case_notes}\nCategory:"
        )
        res = self.router(prompt, max_tokens=25, temperature=0.1)
        category = res["choices"][0]["text"].strip().upper()
        for cat in ["DIFFERENTIAL_DIAGNOSIS", "LITERATURE_ANALYSIS", "PROTOCOL_SYNTHESIS", "FOUNDATIONAL_REVIEW", "GENERAL_CLINICAL"]:
            if cat in category:
                return cat
        return "DIFFERENTIAL_DIAGNOSIS"

    def execute_medical_pipeline(self, case_notes: str):
        task_type = self.route_clinical_task(case_notes)
        print(f"\n[Clinical Router: DeepSeek MoE] Task classified as: {task_type}")

        if task_type == "DIFFERENTIAL_DIAGNOSIS":
            print("[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep clinical differential reasoning...")
            prompt = (
                f"<|im_start|>system\nYou are an expert clinical reasoning assistant governed by strict biomedical validation.<|im_end|>\n"
                f"<|im_start|>user\nAnalyze the following patient presentation and provide a rigorous differential assessment:\n\n{case_notes}<|im_end|>\n"
                f"<|im_start|>assistant\n<think>\n"
            )
            res = self.reasoner(prompt, max_tokens=1536, temperature=0.5, stop=["<|im_end|>"])
            reasoning_output = res["choices"][0]["text"].strip()

            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Formatting structured clinical summary report...")
            synth_prompt = (
                f"<|im_start|>system\nYou are an expert medical writer and clinical synthesizer.<|im_end|>\n"
                f"<|im_start|>user\nBased on the following clinical reasoning, generate a clean, structured medical summary report:\n\n{reasoning_output}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            #final_res = self.synthesizer(synth_prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            final_res = self.synthesizer(synth_prompt, max_tokens=2048, temperature=0.6, stop=["<|im_end|>"])
            return final_res["choices"][0]["text"].strip()

        elif task_type == "LITERATURE_ANALYSIS":
            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Synthesizing biomedical literature and biomarkers...")
            prompt = (
                f"<|im_start|>system\nYou are a biomedical literature analysis assistant.<|im_end|>\n"
                f"<|im_start|>user\n{case_notes}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "PROTOCOL_SYNTHESIS":
            print("[Agent: Sarvam-30B (n_gpu_layers=25)] Processing regional healthcare protocol and guidelines...")
            prompt = f"System: You are an expert clinical protocol processor.\nUser: {case_notes}\nAssistant:"
            res = self.multilingual(prompt, max_tokens=1024, temperature=0.7)
            return res["choices"][0]["text"].strip()

        elif task_type == "FOUNDATIONAL_REVIEW":
            print("[Agent: Certified Scorpion Qwen (n_gpu_layers=-1)] Applying foundational biomedical instruction review...")
            prompt = f"<|im_start|>user\n{case_notes}<|im_end|>\n<|im_start|>assistant\n"
            res = self.foundational(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        else:
            print("[Agent: Qwen Synthesizer] Handling general clinical query...")
            prompt = f"<|im_start|>user\n{case_notes}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

# --- Execution Test ---
if __name__ == "__main__":
    medical_system = PureMedicalAPACAgentSystem()
    sample_clinical_query = "Patient presents with intermittent tachycardia, elevated serum ferritin, and microcytic anemia unresponsive to standard oral iron."
    report = medical_system.execute_medical_pipeline(sample_clinical_query)

    print("\n==============================")
    print("FINAL PURE MEDICAL PIPELINE REPORT")
    print("==============================\n")
    print(report)

Initializing Pure Medical APAC Agent Ecosystem (95GB VRAM Custom Offloading)...
Loading Clinical Router: DeepSeek MoE (n_gpu_layers=-1)...
Loading Clinical Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...
Loading Regional Biomedical Specialist: Sarvam-30B (n_gpu_layers=25)...
Loading Foundational Clinical Model: Certified Scorpion Qwen (n_gpu_layers=-1)...
Loading Deep Clinical Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...
All 5 Medical APAC agent nodes active with custom offloading profiles under Topo-CBP governance.

[Clinical Router: DeepSeek MoE] Task classified as: DIFFERENTIAL_DIAGNOSIS
[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep clinical differential reasoning...
[Agent: Qwen 3.8-27B (n_gpu_layers=25)] Formatting structured clinical summary report...

FINAL PURE MEDICAL PIPELINE REPORT

<think>
The user wants me to take the clinical reasoning provided and generate a clean, structured medical summary report. The reasoning is a detailed differential diagnosis workup f

In [2]:
!nvidia-smi

Sun Nov 15 02:09:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             89W /  600W |   93087MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## EU/USA

In [1]:
!nvidia-smi

Sun Nov 15 04:36:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             49W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import gc
import os
import re
import sys

from llama_cpp import Llama, LlamaGrammar

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)

CATEGORIES = [
    "QUANTITATIVE_MODELING",
    "PORTFOLIO_SYNTHESIS",
    "MACRO_MARKET_ANALYSIS",
    "REGULATORY_COMPLIANCE",
    "GENERAL_FINANCE",
]

FORMAT_RULE = (
    "Write in plain Markdown. Do NOT use LaTeX, \\[ \\], $$ or \\frac. "
    "Write formulas inline in plain text, e.g. VaR = z * sigma * sqrt(t) * V. "
    "Finish every section completely."
)

# MACRO - EU(FRANCE)
# ROUTER,SYTH,FOUNDATION,REASONER - USA

# name -> (file, n_ctx, n_gpu_layers, extra kwargs)
MODELS = {
    "router":     ("gpt-oss-20b-topo-governed-Q4_K_M.gguf",       4096, -1, {}),
    "synth":      ("llama-3.1-70b-topo-governed-q4_k_m.gguf",     8192, 25, {}),
    "macro":      ("mixtral_8x22b_q4_k_m.gguf",                   8192, 25, {}),
    "foundation": ("nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf", 8192, -1, {"flash_attn": True}),
    "reasoner":   ("muse-glimmer-30b-card-faithful-Q4_K_M.gguf",  8192, -1, {}),
}


class PureFinancialGlobalAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models", preload_all=True):
        # preload_all=True : all 5 models stay loaded (your original setup, needs the RAM/VRAM).
        # preload_all=False: only the router stays loaded; the others load on demand.
        # Tip: copy the GGUFs to local disk (/content/models) first; Drive reads are very slow.
        self.model_dir = model_dir
        self.preload_all = preload_all
        self.loaded = {}
        labels = {
            "router": "Financial Router: GPT-OSS-20B (n_gpu_layers=-1)",
            "synth": "Financial Synthesizer: Llama 3.1 70B (n_gpu_layers=25)",
            "macro": "Macro Specialist: Mixtral 8x22B (n_gpu_layers=25)",
            "foundation": "Foundational Risk Model: Nemotron 3.5 Bolt (n_gpu_layers=-1)",
            "reasoner": "Deep Quantitative Reasoner: Muse Glimmer 30B (n_gpu_layers=-1)",
        }
        names = list(MODELS) if preload_all else ["router"]
        for n in names:
            print(f"Loading {labels[n]}...")
            self._load(n)
        if preload_all:
            print("All 5 Financial Western/Global agent nodes active concurrently.")
        else:
            print("Ready. Other models load on demand and are unloaded afterwards.")

    # ---------- model management ----------
    def _load(self, name):
        if name in self.loaded:
            return self.loaded[name]
        fname, n_ctx, n_gpu, extra = MODELS[name]
        self.loaded[name] = Llama(
            model_path=os.path.join(self.model_dir, fname),
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu,
            verbose=False,
            **extra,
        )
        return self.loaded[name]

    def _unload(self, name):
        if self.preload_all:
            return  # keep everything resident
        if name in self.loaded and name != "router":
            del self.loaded[name]
            gc.collect()

    def _unload_all_but_router(self):
        for n in list(self.loaded):
            self._unload(n)

    # ---------- generation helpers ----------
    @staticmethod
    def _chat(llm, system, user, max_tokens, temperature, **kw):
        # create_chat_completion applies the chat template embedded in each GGUF,
        # so Llama 3.1 / Mixtral / GPT-OSS / Nemotron each get their own format.
        res = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            **kw,
        )
        choice = res["choices"][0]
        return choice["message"]["content"].strip(), choice.get("finish_reason")

    @staticmethod
    def _strip_think(text):
        """Return (answer, was_complete)."""
        if "</think>" in text:
            return text.split("</think>", 1)[1].strip(), True
        # Unclosed think block: reasoning was truncated.
        return re.sub(r"^\s*<think>\s*", "", text).strip(), False

    # ---------- routing ----------
    def route_financial_task(self, query: str) -> str:
        router = self._load("router")
        try:
            grammar = LlamaGrammar.from_string(
                "root ::= " + " | ".join(f'"{c}"' for c in CATEGORIES)
            )
            text, _ = self._chat(
                router,
                "Classify the financial query into exactly one category: "
                + ", ".join(CATEGORIES) + ". Reply with the category only.",
                query, max_tokens=16, temperature=0.0, grammar=grammar,
            )
        except Exception as e:
            print(f"[Router] grammar routing failed ({e}); using keyword fallback")
            text = ""
        text = text.upper()
        for c in CATEGORIES:
            if c in text:
                return c
        print(f"[Router] no valid label in {text!r}; falling back to GENERAL_FINANCE")
        return "GENERAL_FINANCE"

    # ---------- pipeline ----------
    def execute_financial_pipeline(self, query: str, task_type: str = None) -> str:
        task = task_type or self.route_financial_task(query)
        print(f"\n[Router: GPT-OSS-20B] Task classified as: {task}")

        if task == "QUANTITATIVE_MODELING":
            print("[Agent: Muse Glimmer 30B] Quantitative reasoning...")
            reasoner = self._load("reasoner")
            raw, finish = self._chat(
                reasoner,
                "You are an expert quantitative finance reasoning assistant. " + FORMAT_RULE,
                "Analyze the following and derive a rigorous quantitative framework "
                f"(assumptions, model, steps, risks):\n\n{query}",
                max_tokens=5000, temperature=0.5,
            )
            answer, complete = self._strip_think(raw)
            if not complete or not answer:
                print("[Warning] Reasoner hit the token limit before finishing; "
                      "using the tail of its reasoning.")
                answer = raw[-6000:]
            self._unload("reasoner")

            print("[Agent: Llama 3.1 70B] Formatting structured report...")
            synth = self._load("synth")
            out, _ = self._chat(
                synth,
                "You are an expert financial analyst and report writer. " + FORMAT_RULE,
                "Turn the following quantitative reasoning into a professional structured "
                "report with sections: Objective, Assumptions, Methodology, "
                "Key Findings, Risks and Limitations, Next Steps.\n\n" + answer,
                max_tokens=2500, temperature=0.4,
            )
            self._unload("synth")
            return out

        if task == "PORTFOLIO_SYNTHESIS":
            print("[Agent: Llama 3.1 70B] Portfolio analysis...")
            llm = self._load("synth")
            out, _ = self._chat(llm, "You are an expert portfolio manager. " + FORMAT_RULE,
                                query, max_tokens=2000, temperature=0.5)
            self._unload("synth")
            return out

        if task == "MACRO_MARKET_ANALYSIS":
            print("[Agent: Mixtral 8x22B] Macro analysis...")
            llm = self._load("macro")
            out, _ = self._chat(llm, "You are an expert global macroeconomic analyst. " + FORMAT_RULE,
                                query, max_tokens=2000, temperature=0.6)
            self._unload("macro")
            return out

        if task == "REGULATORY_COMPLIANCE":
            print("[Agent: Nemotron 3.5 Bolt] Compliance review...")
            llm = self._load("foundation")
            out, _ = self._chat(llm, "You are a financial regulatory compliance analyst. " + FORMAT_RULE,
                                query, max_tokens=2000, temperature=0.4)
            self._unload("foundation")
            return out

        print("[Agent: GPT-OSS-20B] General finance...")
        out, _ = self._chat(self._load("router"), "You are a helpful finance assistant. " + FORMAT_RULE,
                            query, max_tokens=1500, temperature=0.6)
        return out


def save_report(query: str, task: str, body: str, out_dir=None):
    """Save the report as Markdown + a styled HTML page you can open in any browser."""
    import datetime
    import html

    out_dir = out_dir or (
        "/content/drive/MyDrive/reports" if os.path.isdir("/content/drive/MyDrive") else "reports"
    )
    os.makedirs(out_dir, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    base = os.path.join(out_dir, f"financial_report_{stamp}")

    disclaimer = (
        "_AI-generated analysis for research purposes only. Not investment advice. "
        "Verify all figures and model outputs independently before relying on them._"
    )
    md = (
        f"# Financial Analysis Report\n\n"
        f"**Generated:** {datetime.datetime.now():%Y-%m-%d %H:%M}  \n"
        f"**Task type:** {task}  \n"
        f"**Query:** {query}\n\n---\n\n{body}\n\n---\n\n{disclaimer}\n"
    )
    with open(base + ".md", "w", encoding="utf-8") as f:
        f.write(md)

    try:
        import markdown  # pip install markdown
        content = markdown.markdown(md, extensions=["tables", "fenced_code"])
    except ImportError:
        content = "<pre style='white-space:pre-wrap'>" + html.escape(md) + "</pre>"

    page = (
        "<!doctype html><html><head><meta charset='utf-8'><title>Financial Report</title>"
        "<style>body{font-family:Georgia,serif;max-width:820px;margin:40px auto;padding:0 20px;"
        "line-height:1.6;color:#222}h1,h2,h3{font-family:Helvetica,Arial,sans-serif}"
        "table{border-collapse:collapse}td,th{border:1px solid #ccc;padding:6px 10px}"
        "code{background:#f4f4f4;padding:2px 4px}</style></head><body>"
        + content + "</body></html>"
    )
    with open(base + ".html", "w", encoding="utf-8") as f:
        f.write(page)
    return base + ".md", base + ".html"


if __name__ == "__main__":
    system = PureFinancialGlobalAgentSystem()
    q = "Model the impact of cross-border liquidity shocks on multi-currency portfolio Value-at-Risk."
    task = system.route_financial_task(q)
    report = system.execute_financial_pipeline(q, task_type=task)

    md_path, html_path = save_report(q, task, report)

    print("\n==============================")
    print("FINAL PURE FINANCIAL GLOBAL PIPELINE REPORT")
    print("==============================\n")
    print(report)
    print(f"\nSaved report:\n  {md_path}\n  {html_path}")

    # In Colab, render it right in the notebook:
    try:
        from IPython.display import Markdown, display
        display(Markdown(report))
    except Exception:
        pass
    sys.stdout.flush()

Loading Financial Router: GPT-OSS-20B (n_gpu_layers=-1)...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Loading Financial Synthesizer: Llama 3.1 70B (n_gpu_layers=25)...
Loading Macro Specialist: Mixtral 8x22B (n_gpu_layers=25)...
